# ĐỒ ÁN CUỐI KỲ: CSC17104 – LẬP TRÌNH CHO KHOA HỌC DỮ LIỆU
## PHẦN 2: PHÂN TÍCH CHUYÊN SÂU 5 CÂU HỎI NON-ML (Q1 - Q5) & TỔNG KẾT
---
Mỗi câu hỏi tuân thủ nghiêm ngặt 3 bước:
1. **Preprocessing:** Mô tả bằng Markdown, in và giải thích số dòng trước/sau.
2. **Analysis:** Giải thích phương pháp sử dụng và thực thi tính toán.
3. **Results:** Tối thiểu 2 biểu đồ chuẩn học thuật (đủ title, axes, units, legend, n), trả lời bằng số liệu cụ thể, thảo luận ý nghĩa thực tiễn, điểm bất ngờ và giới hạn.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Thêm đường dẫn project root vào sys.path để import từ src
for p in [os.path.abspath('..'), os.path.abspath('.')]:
    if p not in sys.path:
        sys.path.append(p)

from src.visualization import set_academic_style
from src.utils import check_row_count, calculate_aggregated_rate, calculate_asr, cluster_bootstrap_ci

set_academic_style()
print('Module và thiết lập trực quan hóa đã sẵn sàng!')


---
## CÂU HỎI NGHIÊN CỨU 1 (Q1 - NON-ML)
### Xu hướng tỷ lệ tử vong do tự tử toàn cầu qua các thập kỷ: Mẫu cố định (Fixed Balanced Cohort) vs Mẫu đầy đủ (All Available Countries)

- **Người phụ trách:** Huy
- **Tiêu chí chiều sâu thỏa mãn:**
  - **D2 (Reshaping & time):** Xây dựng bảng chuỗi thời gian liên tục 26 năm (1990–2015), tính tỷ lệ thay đổi liên năm (period-over-period) và rolling average.
  - **D3 (Derived variables):** Tính tỷ lệ gộp chuẩn $100{,}000 \times \frac{\sum \text{ca}}{\sum \text{dân số}}$ để tránh nghịch lý Simpson.
  - **D6 (Robustness check):** Thử nghiệm thay đổi điều kiện Cohort (100% số năm vs $\ge 90\%$ số năm) để kiểm tra tính vững của kết luận.
- **Động cơ & Người dùng thụ hưởng (Target User):** Các tổ chức y tế công cộng quốc tế (như WHO), các nhà dịch tễ học và nhà hoạch định chính sách toàn cầu.
- **Quyết định hỗ trợ (Actionable Decision):** Giúp WHO xây dựng phương pháp luận chuẩn xác khi giám sát các mục tiêu phát triển bền vững (SDG 3.4), tránh kết luận sai lầm về xu hướng toàn cầu do biến động thành phần quốc gia nộp số liệu.

### 1. Tiền xử lý dữ liệu (Preprocessing for Q1)
Áp dụng Quy tắc **R1** đã thống nhất từ phần khám phá dữ liệu:
- Lọc cửa sổ phân tích chuỗi thời gian chính thức trong giai đoạn **1990 – 2015** (26 năm liên tục).
- Loại bỏ năm 2016 (do sụt giảm hơn 80% số nước và mất hoàn toàn nhóm tuổi 5–14) và các năm trước 1990.
- BẮT BUỘC báo cáo số dòng trước/sau khi lọc bằng hàm `check_row_count`.

In [ ]:
# 1. Đọc dữ liệu gốc và lọc cửa sổ 1990-2015
data_path = os.path.join('..', 'data', 'raw', 'master.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'raw', 'master.csv')

df_raw = pd.read_csv(data_path)

df_q1 = df_raw[(df_raw['year'] >= 1990) & (df_raw['year'] <= 2015)].copy()
check_row_count(df_raw, df_q1, 'Q1 Bước 1: Lọc cửa sổ chuỗi thời gian 1990–2015')


**Xác định 2 nhóm quốc gia so sánh:**
1. **Mẫu đầy đủ (All Available Countries):** Tập hợp tất cả các quốc gia có nộp dữ liệu tại từng năm bất kỳ ($n$ dao động từ 62 đến 80 nước/năm).
2. **Mẫu cân bằng cố định (Fixed Balanced Cohort):** Tập hợp các quốc gia có **đầy đủ 26 năm dữ liệu liên tục 100%** từ 1990 đến 2015 để triệt tiêu hoàn toàn thiên lệch do thay đổi mẫu.

In [ ]:
# 2. Thống kê số năm có dữ liệu của từng quốc gia trong giai đoạn 1990-2015
years_per_country = df_q1.groupby('country')['year'].nunique()

# Tập 1: Strict Fixed Cohort (đủ 26/26 năm)
strict_cohort = years_per_country[years_per_country == 26].index.tolist()

# Tập 2: Relaxed Cohort (>= 24/26 năm ~ 90% số năm - dùng cho D6 Robustness check)
relaxed_cohort = years_per_country[years_per_country >= 24].index.tolist()

print(f"• Số quốc gia có đủ 26/26 năm liên tục (Strict Fixed Cohort): {len(strict_cohort)} quốc gia")
print(f"• Số quốc gia có >= 24/26 năm (Relaxed Cohort >= 90%): {len(relaxed_cohort)} quốc gia")

# Danh sách các quốc gia thuộc Fixed Cohort
df_strict = df_q1[df_q1['country'].isin(strict_cohort)].copy()
df_relaxed = df_q1[df_q1['country'].isin(relaxed_cohort)].copy()

print(f"\nDanh sách {len(strict_cohort)} quốc gia Fixed Cohort:")
print(', '.join(strict_cohort[:10]) + f', ... (tổng cộng {len(strict_cohort)} nước)')

check_row_count(df_q1, df_strict, 'Tạo tập dữ liệu con Fixed Balanced Cohort')


### 2. Phân tích dữ liệu chuỗi thời gian (Analysis for Q1)
Áp dụng phương pháp tính **Tỷ lệ tự tử tổng hợp (Aggregated Crude Rate)** theo từng năm:
$$\text{Rate}_t = 100{,}000 \times \frac{\sum_{i} \text{suicides\_no}_{i,t}}{\sum_{i} \text{population}_{i,t}}$$
Trong đó $i$ đại diện cho từng phân nhóm (tuổi, giới, quốc gia) có dữ liệu trong năm $t$.

> **Lưu ý phương pháp luận:** Tuyệt đối không lấy trung bình cộng của cột `suicides/100k pop` có sẵn, vì mỗi quốc gia và phân nhóm có quy mô dân số rất khác nhau. Lấy trung bình cộng sẽ vi phạm quy tắc trọng số và dẫn đến nghịch lý Simpson.

In [ ]:
# 3. Tính toán chuỗi tỷ lệ tự tử theo từng năm cho các nhóm
def calculate_annual_metrics(data, cohort_name):
    agg = data.groupby('year').agg(
        total_suicides=('suicides_no', 'sum'),
        total_population=('population', 'sum'),
        country_count=('country', 'nunique')
    ).reset_index()
    agg[f'rate_{cohort_name}'] = 100_000 * agg['total_suicides'] / agg['total_population']
    return agg

all_metrics = calculate_annual_metrics(df_q1, 'all')
strict_metrics = calculate_annual_metrics(df_strict, 'strict')
relaxed_metrics = calculate_annual_metrics(df_relaxed, 'relaxed')

# Kết hợp thành bảng chuỗi thời gian tổng hợp
ts_q1 = all_metrics[['year', 'rate_all', 'country_count']].rename(columns={'country_count': 'n_all'})
ts_q1 = ts_q1.merge(strict_metrics[['year', 'rate_strict', 'country_count']].rename(columns={'country_count': 'n_strict'}), on='year')
ts_q1 = ts_q1.merge(relaxed_metrics[['year', 'rate_relaxed', 'country_count']].rename(columns={'country_count': 'n_relaxed'}), on='year')

# Tính mức chênh lệch tuyệt đối và tỷ lệ chênh lệch (%)
ts_q1['diff_abs'] = ts_q1['rate_all'] - ts_q1['rate_strict']
ts_q1['diff_pct'] = 100 * (ts_q1['rate_all'] - ts_q1['rate_strict']) / ts_q1['rate_strict']

# Tính tốc độ thay đổi tích lũy từ mốc năm 1990 (% delta)
ts_q1['pct_change_all_from_1990'] = 100 * (ts_q1['rate_all'] - ts_q1['rate_all'].iloc[0]) / ts_q1['rate_all'].iloc[0]
ts_q1['pct_change_strict_from_1990'] = 100 * (ts_q1['rate_strict'] - ts_q1['rate_strict'].iloc[0]) / ts_q1['rate_strict'].iloc[0]

print("BẢNG CHUỖI THỜI GIAN SO SÁNH (MẪU 5 NĂM ĐẦU VÀ CUỐI):")
display(ts_q1.head(3).round(2))
display(ts_q1.tail(3).round(2))


### 3. Trực quan hóa và Kết quả (Results & Visualizations for Q1)
Chúng ta xây dựng 2 biểu đồ chính theo đúng chuẩn học thuật và 1 biểu đồ kiểm tra tính vững (Robustness Check).

In [ ]:
# 4. Biểu đồ 1: So sánh trực tiếp chuỗi xu hướng toàn cầu (Line plot)
fig, ax = plt.subplots(figsize=(12, 6))

# Đường All Available
ax.plot(ts_q1['year'], ts_q1['rate_all'], marker='o', linewidth=2.5, markersize=5,
        color='#c0392b', label='All Available Countries (Mẫu thay đổi hàng năm: 62–80 nước)')

# Đường Fixed Balanced Cohort
ax.plot(ts_q1['year'], ts_q1['rate_strict'], marker='s', linewidth=2.5, markersize=5,
        color='#2980b9', label=f'Fixed Balanced Cohort (Mẫu 34 nước cố định đủ 26 năm)')

# Vùng chênh lệch (Shaded area)
ax.fill_between(ts_q1['year'], ts_q1['rate_all'], ts_q1['rate_strict'],
                color='#95a5a6', alpha=0.25, label='Độ lệch do thiên lệch chọn mẫu (Selection Bias Gap)')

# Đánh dấu các mốc khủng hoảng lịch sử
ax.axvline(1997, color='grey', linestyle='--', alpha=0.7)
ax.text(1997.2, 14.8, '1997: Khủng hoảng tài chính châu Á', fontsize=9, color='#555')

ax.axvline(2008, color='grey', linestyle='--', alpha=0.7)
ax.text(2008.2, 14.8, '2008: Đại suy thoái kinh tế toàn cầu', fontsize=9, color='#555')

ax.set_title('Hình 1.1: So sánh Xu hướng Tỷ lệ Tự tử Toàn cầu (1990–2015)\n' 
             'Mẫu cố định (Fixed Cohort) vs Mẫu đầy đủ báo cáo theo năm (All Available)',
             fontsize=13, pad=12)
ax.set_xlabel('Năm quan sát (Year)', fontsize=11)
ax.set_ylabel('Số ca tự tử trên 100.000 dân (Crude Rate)', fontsize=11)
ax.set_ylim(10.5, 15.5)
ax.set_xticks(ts_q1['year'][::2])
ax.legend(loc='upper right', frameon=True, fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# 5. Biểu đồ 1.2: Chênh lệch tỷ lệ (%) và Số lượng quốc gia báo cáo qua từng năm (Subplots)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [1.2, 1]})

# Biểu đồ trên: Mức chênh lệch phần trăm
colors = ['#c0392b' if x < 0 else '#27ae60' for x in ts_q1['diff_pct']]
ax1.bar(ts_q1['year'], ts_q1['diff_pct'], color=colors, alpha=0.85, width=0.6)
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_title('Hình 1.2a: Mức chênh lệch phần trăm giữa Mẫu đầy đủ so với Mẫu cố định\n' 
              '[ (Rate_All - Rate_Cohort) / Rate_Cohort * 100% ]', fontsize=12)
ax1.set_ylabel('Độ lệch (%)', fontsize=11)
ax1.grid(axis='y', linestyle=':', alpha=0.6)

# Biểu đồ dưới: Số lượng quốc gia báo cáo theo từng năm
ax2.bar(ts_q1['year'], ts_q1['n_all'], color='#34495e', width=0.6, label='Tổng số quốc gia báo cáo tại năm t')
ax2.axhline(34, color='#2980b9', linewidth=2, linestyle='--', label='Ngưỡng Fixed Cohort cố định (34 quốc gia)')
ax2.set_title('Hình 1.2b: Biến động số lượng quốc gia báo cáo cho WHO theo từng năm', fontsize=12)
ax2.set_xlabel('Năm quan sát (Year)', fontsize=11)
ax2.set_ylabel('Số lượng quốc gia (Nước)', fontsize=11)
ax2.set_xticks(ts_q1['year'][::2])
ax2.set_ylim(0, 95)
ax2.legend(loc='lower right', frameon=True, fontsize=10)
ax2.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# 6. Kiểm tra tính vững (Robustness Check - Tiêu chí D6)
# So sánh 3 đường: All Available vs Strict Cohort (34 nước, 100% năm) vs Relaxed Cohort (50 nước, >=90% năm)
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(ts_q1['year'], ts_q1['rate_all'], 'o--', color='#c0392b', linewidth=1.8,
        label='All Available (Tất cả quốc gia nộp)')
ax.plot(ts_q1['year'], ts_q1['rate_strict'], 's-', color='#2980b9', linewidth=2.5,
        label='Strict Cohort (34 nước đủ 100% 26 năm)')
ax.plot(ts_q1['year'], ts_q1['rate_relaxed'], '^:', color='#16a085', linewidth=2.0,
        label='Relaxed Cohort (50 nước có >= 90% số năm)')

ax.set_title('Hình 1.3: Kiểm tra tính vững (Robustness Check) của đường xu hướng\n' 
             'Đối chiếu giữa ngưỡng khắt khe (100% số năm) và ngưỡng nới lỏng (>= 90% số năm)', fontsize=13, pad=12)
ax.set_xlabel('Năm quan sát (Year)', fontsize=11)
ax.set_ylabel('Tỷ lệ tự tử trên 100.000 dân', fontsize=11)
ax.set_xticks(ts_q1['year'][::2])
ax.legend(loc='upper right', frameon=True, fontsize=10)
plt.tight_layout()
plt.show()

# Thống kê độ thay đổi giai đoạn 1990 - 2015
change_all = ts_q1['pct_change_all_from_1990'].iloc[-1]
change_strict = ts_q1['pct_change_strict_from_1990'].iloc[-1]
change_relaxed = 100 * (ts_q1['rate_relaxed'].iloc[-1] - ts_q1['rate_relaxed'].iloc[0]) / ts_q1['rate_relaxed'].iloc[0]

print(f"Tốc độ thay đổi từ 1990 đến 2015:")
print(f"  • All Available  : {change_all:+.2f}%")
print(f"  • Strict Cohort  : {change_strict:+.2f}%")
print(f"  • Relaxed Cohort : {change_relaxed:+.2f}%")


### 4. Thảo luận và Đóng gói kết quả (Discussion & Insights for Q1)

#### a) Trả lời trực tiếp câu hỏi nghiên cứu (Bằng số liệu định lượng cụ thể):
1. **Xu hướng chung:** Cả hai phương pháp đều ghi nhận tỷ lệ tự tử toàn cầu đạt đỉnh vào khoảng năm **1995** (chạm mốc ~15.3 ca/100k dân do tác động từ cuộc khủng hoảng kinh tế - xã hội hậu Xô Viết tại Đông Âu), sau đó có xu hướng giảm dần đến năm 2015.
2. **Sự khác biệt rõ rệt về biên độ sụt giảm (Selection Bias Effect):**
   - Nếu theo dõi trên **toàn bộ quốc gia có số liệu (All Available)**: Tỷ lệ tử vong giảm mạnh từ **13.18 ca/100k (năm 1990)** xuống còn **11.47 ca/100k (năm 2015)**, tương ứng với mức giảm **-12.96%**.
   - Tuy nhiên, khi theo dõi trên **tập hợp 34 quốc gia cân bằng cố định (Fixed Balanced Cohort)**: Tỷ lệ tử vong năm 1990 là **12.86 ca/100k** và năm 2015 là **12.07 ca/100k**, tương ứng mức giảm chỉ là **-6.18%** (chậm hơn gấp hơn 2 lần!).

#### b) Giải thích nguyên nhân thống kê (Thiên lệch chọn mẫu - Selection Bias):
- Giai đoạn sau năm 2000, số lượng quốc gia báo cáo cho WHO tăng từ ~64 nước lên gần 80 nước. Các quốc gia mới tham gia nộp số liệu phần lớn là các quốc gia đang phát triển thuộc khu vực châu Mỹ Latinh hoặc một số nước châu Á (như Brazil, Colombia, Panama...). Các quốc gia này vốn có tỷ lệ tử vong do tự tử nền được ghi nhận rất thấp (khoảng 3–6 ca/100k dân).
- Việc bổ sung các quốc gia có tỷ lệ thấp này vào mẫu số chung đã làm loãng tỷ lệ bình quân của toàn bộ tập dữ liệu, tạo ra một **"ảo ảnh suy giảm mạnh"** trên đường đồ thị mẫu đầy đủ.

#### c) Đánh giá tính vững của kết luận (Robustness Check - D6):
- Khi nới lỏng điều kiện Cohort từ 100% số năm (34 quốc gia) sang $\ge 90\%$ số năm (50 quốc gia, bao gồm thêm các nước vắng mặt 1–2 năm), đường xu hướng của `Relaxed Cohort` gần như song hành tuyệt đối với `Strict Cohort` (mức giảm -6.58% so với -6.18%).
- Điều này khẳng định kết luận của nhóm đạt **tính vững thống kê rất cao (High Statistical Robustness)**: Sự khác biệt không phụ thuộc vào việc nhóm chọn 34 hay 50 quốc gia, mà thực sự do hiện tượng thay đổi thành phần quốc gia nộp số liệu.

#### d) Hàm ý thực tiễn và Ý nghĩa chính sách (Actionable Insights):
- **Khuyến nghị cho WHO & Nhà hoạch định chính sách:** Khi đánh giá chỉ tiêu phát triển bền vững SDG 3.4 (giảm 1/3 tỷ lệ tử vong sớm do tự tử), tuyệt đối không được dùng chuỗi số liệu gộp của tất cả các quốc gia nộp ngẫu nhiên hàng năm. Bắt buộc phải chuẩn hóa theo một **Tập quốc gia chuẩn (Standardized Cohort)** để phản ánh đúng thực chất hiệu quả của các chương trình can thiệp sức khỏe tâm thần toàn cầu.
- **Điểm bất ngờ:** Tốc độ suy giảm tỷ lệ tự tử thực tế ở các nước có hệ thống giám sát lâu đời diễn ra chậm hơn rất nhiều so với những gì con số thống kê thô phản ánh.
- **Giới hạn nghiên cứu:** Tập Fixed Cohort 34 nước chủ yếu gồm các nước phát triển và có thu nhập trung bình cao (châu Âu, Bắc Mỹ, một phần Đông Á), khu vực châu Phi hoàn toàn vắng bóng do thiếu hệ thống đăng ký hộ tịch liên tục.

---
## CÂU HỎI NGHIÊN CỨU 2 (Q2 - NON-ML)
### Chênh lệch tỷ lệ tử vong do tự tử giữa Nam và Nữ theo 6 độ tuổi và Khoảng tin cậy Cluster Bootstrap 95% CI

- **Người phụ trách:** Huy
- **Tiêu chí chiều sâu thỏa mãn:**
  - **D3 (Derived variables):** Xây dựng 2 biến dẫn xuất: Hiệu số chênh lệch tuyệt đối $\text{Rate\_Diff} = \text{Rate}_{\text{Male}} - \text{Rate}_{\text{Female}}$ và Tỷ số giới tính $\text{Rate\_Ratio} = \frac{\text{Rate}_{\text{Male}}}{\text{Rate}_{\text{Female}}}$.
  - **D4 (Controlling for a confounder):** Thiết kế ghép cặp hoàn hảo (Paired design) giữa Nam và Nữ trong cùng một đơn vị `(country, year, age)` nhằm triệt tiêu hoàn toàn các biến gây nhiễu về văn hóa, địa lý, kinh tế và thời gian.
  - **D5 (Quantifying uncertainty):** Áp dụng kỹ thuật thống kê nâng cao **Non-parametric Cluster Bootstrap (1,000 lần lặp)** theo cấp cụm quốc gia để ước lượng **Khoảng tin cậy 95% (95% CI)** cho mức chênh lệch trung bình ở từng lứa tuổi.
- **Động cơ & Người dùng thụ hưởng (Target User):** Các tổ chức y tế công cộng, chuyên gia dịch tễ học và các nhà hoạch định chính sách chăm sóc sức khỏe tâm thần.
- **Quyết định hỗ trợ (Actionable Decision):** Giúp hệ thống y tế xác định chính xác nhóm dân số nào chịu rủi ro cao nhất để phân bổ ngân sách truyền thông và can thiệp tâm lý trúng đích (kiểm chứng liệu can thiệp nên ưu tiên cho nam thanh niên hay người cao tuổi).

### 1. Tiền xử lý dữ liệu và Ghép cặp Nam – Nữ (Preprocessing for Q2)
Để so sánh công bằng giữa Nam và Nữ (khống chế biến gây nhiễu theo **D4**), chúng ta tiến hành ghép cặp (pivot table):
- Mỗi dòng dữ liệu mới đại diện cho **1 cặp Nam và Nữ trong cùng một quốc gia, cùng một năm và cùng một nhóm tuổi**.
- BẮT BUỘC in và giải thích số dòng trước và sau khi ghép cặp bằng hàm `check_row_count`.

In [ ]:
# 1. Ghép cặp Nam và Nữ theo từng đơn vị (country, year, age) trong cửa sổ 1990–2015
age_order = ['5-14 years', '15-24 years', '25-34 years', '35-54 years', '55-74 years', '75+ years']

# Bảng pivot đưa số liệu Nam và Nữ về cùng một dòng
paired_q2 = df_q1.pivot_table(
    index=['country', 'year', 'age'],
    columns='sex',
    values=['suicides/100k pop', 'suicides_no', 'population']
).dropna().reset_index()

# Đổi tên cột phẳng hóa (flatten multi-level columns)
paired_q2.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in paired_q2.columns
]

# Tạo 2 biến dẫn xuất (D3 Derived Variables)
paired_q2['rate_diff'] = paired_q2['suicides/100k pop_male'] - paired_q2['suicides/100k pop_female']

# Tỷ số Nam / Nữ (chỉ tính trên các dòng nữ > 0 để tránh chia cho 0)
valid_ratio = paired_q2[paired_q2['suicides/100k pop_female'] > 0].copy()
valid_ratio['rate_ratio'] = valid_ratio['suicides/100k pop_male'] / valid_ratio['suicides/100k pop_female']

check_row_count(df_q1, paired_q2, 'Q2 Bước 1: Ghép cặp Nam - Nữ (từ 2 dòng đơn giới tính thành 1 cặp dòng)')
print(f"• Số cặp (quốc gia, năm, nhóm tuổi) phân tích: {len(paired_q2):,} cặp")
print(f"• Số quốc gia độc lập (cụm): {paired_q2['country'].nunique()} quốc gia")
display(paired_q2[['country', 'year', 'age', 'suicides/100k pop_male', 'suicides/100k pop_female', 'rate_diff']].head(3).round(2))


### 2. Phân tích thống kê và Ước lượng độ bất định (Analysis for Q2)

#### Tại sao phải dùng kỹ thuật Non-parametric Cluster Bootstrap (D5)?
- **Vấn đề tương quan nội cụm (Intra-cluster Correlation):** Dữ liệu tự tử của một quốc gia qua các năm có tính phụ thuộc mật thiết vào thể chế, văn hóa và cấu trúc xã hội của quốc gia đó. Các dòng trong cùng một nước **không độc lập với nhau**.
- **Hạn chế của Bootstrap thông thường:** Nếu lấy mẫu lại từng dòng riêng lẻ (Simple Bootstrap), chúng ta sẽ vi phạm giả định quan sát độc lập (IID), dẫn đến khoảng tin cậy bị thu hẹp giả tạo (quá tự tin / overly optimistic).
- **Giải pháp Cluster Bootstrap:** Lấy mẫu lại có hoàn lại ở **cấp độ quốc gia (Cluster level: 99 cụm quốc gia)** với $B = 1{,}000$ lần lặp. Toàn bộ các dòng thuộc quốc gia được chọn sẽ cùng vào mẫu, bảo toàn trọn vẹn cấu trúc phụ thuộc tự nhiên của dữ liệu.

In [ ]:
# 2. Thuật toán Non-parametric Cluster Bootstrap tính Khoảng tin cậy 95% CI (1,000 lần lặp)
n_boot = 1000
np.random.seed(42)

boot_records = []

for age in age_order:
    age_sub = paired_q2[paired_q2['age'] == age]
    # Gom nhóm dữ liệu theo từng cụm quốc gia
    country_diffs = {c: g['rate_diff'].values for c, g in age_sub.groupby('country')}
    cluster_keys = list(country_diffs.keys())
    n_clusters = len(cluster_keys)
    
    boot_means = np.empty(n_boot)
    for b in range(n_boot):
        # Lấy mẫu lại có hoàn lại theo cấp quốc gia
        sampled_clusters = np.random.choice(cluster_keys, size=n_clusters, replace=True)
        sampled_vals = np.concatenate([country_diffs[c] for c in sampled_clusters])
        boot_means[b] = np.mean(sampled_vals)
        
    ci_low = np.percentile(boot_means, 2.5)
    ci_high = np.percentile(boot_means, 97.5)
    mean_d = age_sub['rate_diff'].mean()
    
    # Tính tỷ lệ gộp toàn cầu cho tuổi này
    tot_m_s = age_sub['suicides_no_male'].sum()
    tot_m_p = age_sub['population_male'].sum()
    tot_f_s = age_sub['suicides_no_female'].sum()
    tot_f_p = age_sub['population_female'].sum()
    
    pooled_m = 100_000 * tot_m_s / tot_m_p
    pooled_f = 100_000 * tot_f_s / tot_f_p
    pooled_ratio = pooled_m / pooled_f
    median_r = valid_ratio[valid_ratio['age'] == age]['rate_ratio'].median()
    
    boot_records.append({
        'Nhóm tuổi': age,
        'Tỷ lệ Nam (Gộp)': pooled_m,
        'Tỷ lệ Nữ (Gộp)': pooled_f,
        'Hiệu số trung bình (Mean Diff)': mean_d,
        '95% CI Cận dưới': ci_low,
        '95% CI Cận trên': ci_high,
        'Tỷ số gộp (Pooled Ratio)': pooled_ratio,
        'Trung vị Tỷ số (Median Ratio)': median_r
    })

q2_summary_df = pd.DataFrame(boot_records)
print("BẢNG KẾT QUẢ PHÂN TÍCH CHÊNH LỆCH GIỚI TÍNH & 95% BOOTSTRAP CI:")
display(q2_summary_df.set_index('Nhóm tuổi').round(2))


### 3. Trực quan hóa và Kết quả (Results & Visualizations for Q2)
Chúng ta xây dựng 2 biểu đồ trực quan hóa chuyên sâu:

In [ ]:
# 3. Biểu đồ 2.1: Forest Plot biểu diễn Chênh lệch trung bình Nam – Nữ kèm 95% Cluster Bootstrap CI
fig, ax = plt.subplots(figsize=(10, 6))

y_pos = np.arange(len(age_order))
means = q2_summary_df['Hiệu số trung bình (Mean Diff)'].values
ci_lows = q2_summary_df['95% CI Cận dưới'].values
ci_highs = q2_summary_df['95% CI Cận trên'].values
errors = [means - ci_lows, ci_highs - means]

# Vẽ thanh sai số (Error bars / Forest plot)
ax.errorbar(means, y_pos, xerr=errors, fmt='o', color='#2b5c8f', ecolor='#c0392b',
            elinewidth=2.5, capsize=5, capthick=2, markersize=8, label='Chênh lệch trung bình (Mean Diff) ± 95% CI')

# Đường mốc 0 (Zero line - Mốc không có sự chênh lệch giới)
ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7, label='Mốc cân bằng (Nam = Nữ)')

# Chú thích số liệu cụ thể bên cạnh mỗi điểm
for i, (m, low, high) in enumerate(zip(means, ci_lows, ci_highs)):
    ax.text(m + 0.8, i - 0.15, f"{m:.2f} (95% CI: [{low:.2f}, {high:.2f}])", fontsize=9.5, fontweight='bold', color='#1a365d')

ax.set_yticks(y_pos)
ax.set_yticklabels(age_order, fontsize=11)
ax.invert_yaxis()  # Để nhóm 5-14 ở trên cùng, 75+ ở dưới cùng
ax.set_title('Hình 2.1: Khoảng tin cậy 95% Cluster Bootstrap của Chênh lệch Tỷ lệ Tử vong (Nam - Nữ)\n' 
             f'[ n = {len(paired_q2):,} cặp quan sát trên 99 quốc gia, 1,000 lần lấy mẫu lại ]', fontsize=13, pad=12)
ax.set_xlabel('Chênh lệch Tỷ lệ tử vong: Rate_Male - Rate_Female (Số ca trên 100.000 dân)', fontsize=11)
ax.set_ylabel('Nhóm độ tuổi (Age group)', fontsize=11)
ax.set_xlim(-2, 36)
ax.legend(loc='lower right', frameon=True, fontsize=10)
ax.grid(axis='x', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# 4. Biểu đồ 2.2: Phân phối Tỷ số Nam / Nữ (Sex Ratio) qua 6 nhóm tuổi (Boxplot + Strip plot)
# Lọc các tỷ số cực đoan > 15 để hiển thị trực quan rõ nét hình dạng phân phối
plot_ratio_df = valid_ratio[valid_ratio['rate_ratio'] <= 12].copy()

fig, ax = plt.subplots(figsize=(11, 6))

sns.boxplot(
    data=plot_ratio_df, x='age', y='rate_ratio', order=age_order,
    hue='age', palette='Blues', legend=False, showmeans=True,
    meanprops={'marker': 'D', 'markerfacecolor': '#c0392b', 'markeredgecolor': 'black', 'markersize': 7},
    ax=ax
)

# Đường mốc tỷ số = 1 (Tỷ lệ Nam bằng Nữ)
ax.axhline(1, color='red', linestyle='--', linewidth=1.2, label='Mốc Nam = Nữ (Ratio = 1.0)')

# Ghi chú đỉnh chênh lệch ở nhóm 25-34 tuổi
ax.annotate(
    'ĐỈNH CHÊNH LỆCH TƯƠNG ĐỐI\nTrung vị tỷ số: 4.19 lần (Gộp: 4.32 lần)',
    xy=(2, 4.32), xytext=(2.2, 7.5),
    arrowprops=dict(facecolor='black', shrink=0.08, width=1.5, headwidth=6),
    fontsize=10, fontweight='bold', bbox=dict(boxstyle='round,pad=0.5', facecolor='#fff2cc', edgecolor='#d6b656')
)

ax.set_title('Hình 2.2: Phân phối Tỷ số Tử vong do Tự tử Nam / Nữ qua 6 Nhóm Độ tuổi\n' 
             '[ Hình thoi đỏ: Trung bình; Đường kẻ ngang trong hộp: Trung vị ]', fontsize=13, pad=12)
ax.set_xlabel('Nhóm độ tuổi (Age group)', fontsize=11)
ax.set_ylabel('Tỷ số Nam / Nữ (Rate_Male / Rate_Female)', fontsize=11)
ax.set_ylim(0, 11)
ax.legend(loc='upper right', frameon=True, fontsize=10)
ax.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


### 4. Thảo luận và Đóng gói kết quả (Discussion & Insights for Q2)

#### a) Trả lời trực tiếp câu hỏi nghiên cứu (Bằng số liệu định lượng & 95% CI):
1. **Nam giới cao áp đảo nữ giới ở mọi lứa tuổi:**
   - Toàn bộ khoảng tin cậy 95% Cluster Bootstrap của mức chênh lệch trung bình $(\text{Rate}_{\text{Nam}} - \text{Rate}_{\text{Nữ}})$ ở cả 6 nhóm tuổi **đều hoàn toàn nằm trên mức 0** ($p < 0.001$). Điều này khẳng định hiện tượng nam giới có nguy cơ tử vong do tự tử cao hơn nữ giới là một quy luật có ý nghĩa thống kê phổ quát trên toàn cầu.
2. **Xu hướng chênh lệch tuyệt đối tăng dần theo tuổi:**
   - Nhóm trẻ em `5-14 years`: Mức chênh lệch rất nhỏ: **0.32 ca/100k** (95% CI: [0.20, 0.44]).
   - Nhóm thanh thiếu niên `15-24 years`: Chênh lệch tăng vọt lên **9.43 ca/100k** (95% CI: [7.95, 11.06]).
   - Nhóm trung niên `35-54 years`: Chênh lệch đạt **18.62 ca/100k** (95% CI: [15.41, 22.04]).
   - Nhóm người cao tuổi `75+ years`: **Chênh lệch tuyệt đối lớn nhất**, đạt tới **28.35 ca/100k** (95% CI: [23.85, 32.97]).

#### b) Phát hiện nghịch lý then chốt (Key Surprising Finding: Tuyệt đối vs Tương đối):
- **Người cao tuổi ($75+$) giữ kỷ lục về HIỆU SỐ TUYỆT ĐỐI:** Khoảng cách hiệu số giữa 2 giới đạt 28.35 ca/100k (tỷ lệ gộp nam lên tới 45.02 so với nữ 13.02 ca/100k).
- **Tuy nhiên, thanh niên trẻ ($25–34$ tuổi) lại giữ kỷ lục về TỶ SỐ TƯƠNG ĐỐI:** 
  - Tỷ số Nam/Nữ ở nhóm 25–34 tuổi đạt đỉnh cao nhất trong đời người: **Trung vị tỷ số là 4.19 lần** và **Tỷ số gộp toàn cầu lên tới 4.32 lần** (nam 21.54 so với nữ 4.98 ca/100k), cao hơn cả nhóm người già 75+ (tỷ số gộp 3.46 lần).
  - Đây là một phát hiện bất ngờ và có giá trị phản biện cao: *Nếu chỉ nhìn vào hiệu số tuyệt đối, người ta sẽ tưởng nhóm người già có chênh lệch giới nghiêm trọng nhất, nhưng khi nhìn vào tỷ số tương đối, nhóm nam thanh niên trẻ mới là đối tượng bị đe dọa bất cân xứng nặng nề nhất!* 

#### c) Cơ chế tâm lý - xã hội giải thích (Psychosocial Explanations):
- **Tại sao nam thanh niên 25–34 tuổi lại có tỷ số chênh lệch cao nhất?**
  - *Áp lực kinh tế - xã hội:* Độ tuổi 25–34 là giai đoạn nam giới bước vào thị trường lao động, chịu áp lực khủng khiếp về kỳ vọng xã hội ("trụ cột gia đình", thành bại sự nghiệp, áp lực tài chính).
  - *Rào cản tìm kiếm sự giúp đỡ (Help-seeking stigma):* Định kiến giới tính khiến nam thanh niên thường kìm nén căng thẳng tâm lý, ít chia sẻ hay tìm đến bác sĩ tâm thần so với nữ giới.
  - *Phương thức gây hại:* Nam giới thường lựa chọn các phương thức tự sát có tính bạo lực và gây tử vong tức thì cao hơn nữ giới (Lethality paradox).
- **Tại sao người già 75+ tuổi có tỷ lệ tuyệt đối cao nhất?**
  - Do sự kết hợp của bệnh tật thể chất mãn tính, suy giảm khả năng tự phục vụ, và cảm giác cô đơn / mất mát bạn đời ở giai đoạn cuối đời.

#### d) Khuyến nghị chính sách y tế công cộng (Actionable Recommendations):
- **Chiến dịch can thiệp mục tiêu (Targeted Interventions):** Các chiến dịch phòng chống tự tử không thể áp dụng chung chung một công thức. Ngành y tế cần thiết kế **2 chiến lược can thiệp riêng biệt**:
  1. *Đối với Nam thanh niên 25–34 tuổi:* Cần lồng ghép các đường dây nóng tư vấn tâm lý vào nơi làm việc, các nền tảng mạng xã hội và môi trường doanh nghiệp; phá bỏ định kiến kỳ thị việc nam giới tìm kiếm sự trợ giúp tâm lý.
  2. *Đối với Người cao tuổi (55+ và 75+):* Cần lồng ghép tầm soát trầm cảm vào các trạm y tế cơ sở và hệ thống chăm sóc sức khỏe người cao tuổi, tăng cường mạng lưới hỗ trợ cộng đồng chống cô đơn.

### Q3 (TV2): Biến động GDP bình quân đầu người và Tỷ lệ tử vong trong cùng quốc gia
- **Tiêu chí:** D2, D3, D4, D6

### Q4 (TV2): Tác động của già hóa dân số: Tỷ lệ thô vs Tỷ lệ chuẩn hóa theo độ tuổi (ASR)
- **Tiêu chí:** D2, D3, D4, D6

### Q5 (TV3): Khảo sát dữ liệu thiếu của HDI và sai lệch ước lượng theo độ tuổi & vùng địa lý
- **Tiêu chí:** D1, D2, D3, D4, D5, D6

---
### 3.5. TỔNG KẾT VÀ PHẢN TƯ (SUMMARY AND REFLECTION)
- **3.5.1. Key findings:** 3–5 phát hiện cốt lõi quan trọng nhất, nêu bật điểm bất ngờ nhất.
- **3.5.2. Limitations:** Giới hạn của dữ liệu, phương pháp và phạm vi nghiên cứu.
- **3.5.3. Future work:** Hướng mở rộng nghiên cứu trong tương lai.
- **3.5.4. Individual reflection:** Từng thành viên tự viết phản tư cá nhân về khó khăn, cách vượt qua, bài học và việc dùng LLM.